In [ ]:
from robovast.common.analysis import read_output_csv, read_runs
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = ''

# One table per run, written into the run directory by the `resource_usage` postprocessing
# step: every container's samples, already cut to this run's share of its job's timeline.
#
# The container is a COLUMN, so nothing here names the containers a campaign happens to
# have. The previous version read resource_usage_main/nav/simulation.csv by name, from the
# run directory where those job artifacts have never been -- so it found nothing, and its
# bare `except` reported that as "no resource usage data" rather than as a missing file.
try:
    df = read_output_csv(DATA_DIR, "resource_usage.csv")
except FileNotFoundError as e:
# A campaign postprocessed BEFORE the resource_usage step existed has no such table --
# the step is auto-injected, so this is a question of when the campaign was processed,
# not of how it was configured. Re-running postprocessing produces it from the raw
# per-container CSVs, which are kept in _jobs/ forever, so nothing is re-executed.
    raise RuntimeError(
        f"{e}\n\nThis run has no resource_usage table. Re-run postprocessing on this "
        f"campaign to produce it; the raw per-container CSVs under _jobs/ are kept, so "
        f"no run has to be executed again."
    ) from e

# `timestamp` is SIMULATION seconds, which is what makes two simulators comparable; it is
# empty for every sample taken before the simulator began publishing /clock (image boot,
# stack bring-up). Those are real measurements with no sim time, so say how many rather
# than dropping them silently.
before_clock = int(df['timestamp'].isna().sum())
if before_clock:
    print(f"{before_clock} of {len(df)} samples predate /clock (bring-up) "
          f"and are not on the sim-time axis.")

# in_window == 0 is bring-up and teardown attributed to this run, not the trial itself.
trial = df[df['in_window'] == 1]
print(f"{len(trial)} samples in the trial, across {trial['container'].nunique()} container(s): "
      f"{', '.join(sorted(trial['container'].unique()))}")


In [ ]:
# Load per container over time. One row is one PROCESS NAME, so the container's load is the
# SUM over its processes at that instant -- a mean here would silently report a per-process
# figure instead.
overall = (trial.dropna(subset=['timestamp'])
                .groupby(['container', 'timestamp'], as_index=False)
                .agg(cpu_percent=('cpu_percent', 'sum'),
                     memory_rss_bytes=('memory_rss_bytes', 'sum')))
overall['mem_usage_mb'] = overall['memory_rss_bytes'] / (1024 * 1024)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for container, group in overall.groupby('container'):
    group = group.sort_values('timestamp')
    ax1.plot(group['timestamp'], group['cpu_percent'], linewidth=2, label=container)
    ax2.plot(group['timestamp'], group['mem_usage_mb'], linewidth=2, label=container)

# cpu_percent is psutil's PER-CORE percentage, so one saturated core is 100 and the ceiling
# is 100 * available_cpus. Drawing that line is the whole point of the plot: a container
# pinned against it is a candidate explanation for whatever the run did.
# From the `runs` table, not from job/sysinfo.yaml. The YAML writes every field as a
# string, so `100.0 * cpus` raised TypeError on a value that `if cpus:` had already
# accepted -- and the bare `except` around the read did not cover the arithmetic. The
# ingest types this column as REAL, which is what the ceiling has to be multiplied by.
cpus = None
_runs = read_runs(DATA_DIR)
if 'available_cpus' in _runs.columns:
    _cpus = _runs['available_cpus'].dropna()
    cpus = float(_cpus.iloc[0]) if len(_cpus) else None
if cpus is None:
    print("This run records no host CPU count, so the saturation ceiling is not drawn.")
if cpus:
    ax1.axhline(100.0 * cpus, color='crimson', linestyle='--', linewidth=1,
                label=f'saturation ({cpus} cores)')

ax1.set_ylabel('CPU (%, per-core: 100 = one core)', fontsize=12)
ax1.set_title('CPU per container over the trial', fontsize=14, fontweight='bold')
ax2.set_ylabel('Memory RSS (MB, sums shared pages)', fontsize=12)
ax2.set_xlabel('Simulation time (s)', fontsize=12)
ax2.set_title('Memory per container over the trial', fontsize=14, fontweight='bold')
for ax in (ax1, ax2):
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np


def plot_pie_charts(group, container):
    """What inside this container was heavy. `name` is the process name, which is the stable
    key -- pids churn across a run, so a per-pid breakdown would split one node in two."""
    process_usage = (group.groupby('name', as_index=False)
                          .agg(cpu_percent=('cpu_percent', 'mean'),
                               memory_rss_bytes=('memory_rss_bytes', 'mean')))
    process_usage['mem_usage_mb'] = process_usage['memory_rss_bytes'] / (1024 * 1024)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle(container, fontsize=16, fontweight='bold')

    def make_pie(ax, column, threshold, unit, title):
        data = process_usage.sort_values(column, ascending=False)
        significant = data[data[column] > threshold]
        other = data[data[column] <= threshold][column].sum()
        if other > 0:
            plot_data = pd.concat([
                significant[['name', column]],
                pd.DataFrame([{'name': 'Other', column: other}]),
            ], ignore_index=True)
        else:
            plot_data = significant[['name', column]].copy()
        if plot_data.empty:
            ax.set_axis_off()
            return
        colors = plt.cm.Set3(np.linspace(0, 1, len(plot_data)))
        _, _, autotexts = ax.pie(plot_data[column], labels=plot_data['name'],
                                 autopct='%1.1f%%', startangle=90, colors=colors)
        ax.set_title(f'{title}\n(threshold > {threshold} {unit})',
                     fontsize=13, fontweight='bold')
        for at in autotexts:
            at.set_fontweight('bold')
            at.set_fontsize(9)

    make_pie(ax1, 'cpu_percent', threshold=0.5, unit='%', title='Mean CPU by process')
    make_pie(ax2, 'mem_usage_mb', threshold=50, unit='MB', title='Mean memory by process')

    plt.tight_layout()
    plt.show()


for container, group in trial.groupby('container'):
    plot_pie_charts(group, container)
